In [1]:
import json
from pathlib import Path

In [2]:

import pandas as pd

In [1]:
from src.setup.structure import parameters_from_tar_filename

In [4]:
DATA_ROOT = Path("./data/res")
DEX_SORT = DATA_ROOT / "dex_sort.json"

In [5]:
def show_full_df(df: pd.DataFrame):
    with pd.option_context(
        "display.max_rows", None, "display.max_columns", None
    ):  # more options can be specified also
        display(df)

# Load DEX-Sort data

In [6]:
with open(DEX_SORT, "r") as file:
    data = json.load(file)

# Flatten the data into a format we can use for DataFrame creation
flattened_data = []

# Iterate through each entry
for filename, file_data in data.items():
    # Process the 'playstore' and 'local' dictionaries
    for category in ["playstore", "local"]:
        for hash_key, dex_file in file_data.get(category, {}).items():
            flattened_data.append(
                {
                    "filename": filename,
                    "category": category,
                    "hash_key": hash_key,
                    "dex_file": dex_file,
                }
            )

    # Process the 'differing_dexes' dictionary
    for hash_key, differing_info in file_data.get("differing_dexes", {}).items():
        flattened_data.append(
            {
                "filename": filename,
                "category": "differing_dexes",
                "hash_key": hash_key,
                "dex_file": differing_info,
            }
        )

# Convert the flattened data into a pandas DataFrame
df = pd.DataFrame(flattened_data)

# Set 'filename', 'category', and 'hash_key' as the index
df.set_index(["filename", "category", "hash_key"], inplace=True)

# dex_files grouped by filename > category > sha256 hash
df.head(14)

dex_file
filename                      category        hash_key                                                                   
signal-android_v7.30.2.tar.gz playstore       a4decdb3a060d81ecc3f59f8ee02e625629ed2b17546d96...             classes3.dex
                                              c061b0748ea8654fa3a585f4e11557e35889517eecb732f...             classes4.dex
                                              5b2bd0e9dce85b78981212f3b41cd55e87d0c3c0017911a...              classes.dex
                                              98059dae55a3c4c379e9a32f1b38b8a347bd6fb39ed87b0...             classes7.dex
                                              f32782128a6145b054f1a5c65cb05902ed002e71033ce1f...             classes2.dex
                                              d6ca0fdf8a5914c6a2f302756f497c2e057e6482eafabc0...             classes5.dex
                                              4d8d31008edf2ebb087e457c115b6f76878b6af3ab0aa8d...             classes6.dex
                              local           f32782128a6145b054f1a5c65cb05902ed002e71033ce1f...             classes3.dex
                                              a4decdb3a060d81ecc3f59f8ee02e625629ed2b17546d96...             classes4.dex
                                              5b2bd0e9dce85b78981212f3b41cd55e87d0c3c0017911a...              classes.dex
                                              4d8d31008edf2ebb087e457c115b6f76878b6af3ab0aa8d...             classes2.dex
                                              d6ca0fdf8a5914c6a2f302756f497c2e057e6482eafabc0...             classes5.dex
                                              c061b0748ea8654fa3a585f4e11557e35889517eecb732f...             classes6.dex
                              differing_dexes 98059dae55a3c4c379e9a32f1b38b8a347bd6fb39ed87b0...  playstore->classes7.dex

In [7]:
df.describe()

,dex_file
count,398
unique,20
top,classes3.dex
freq,50


# Check 1: Files with the same hash, and maybe different name

In [8]:
def find_differing_file_names(df, sort_keys=True):
    """
    Given a dataframe entry, do an inner join on the sha256
    """
    # Extract rows for playstore and local categories
    playstore_df = df[df.index.get_level_values("category") == "playstore"]
    local_df = df[df.index.get_level_values("category") == "local"]

    # Merge the playstore and local dataframes on the hash_key
    merged_df = playstore_df.merge(
        local_df, on="hash_key", suffixes=("_playstore", "_local"), how="inner"
    )

    # Find rows where dex_file names are different between playstore and local
    differing_files = merged_df[
        merged_df["dex_file_playstore"] != merged_df["dex_file_local"]
    ]

    # Return the differing files with relevant info
    if sort_keys:
        return differing_files[["dex_file_playstore", "dex_file_local"]].sort_values(
            by=["dex_file_playstore"]
        )
    else:
        return differing_files[["dex_file_playstore", "dex_file_local"]]

In [9]:
# Initialize an empty list to store the results
all_differing_files = []

# Iterate over each filename in the DataFrame and find differing files
for filename in df.index.get_level_values("filename").unique():
    differing_files_df = find_differing_file_names(df.loc[filename])
    differing_files_df["filename"] = filename  # Add filename column
    all_differing_files.append(differing_files_df)

# Combine the results into a single DataFrame
dex_mapping_df = pd.concat(all_differing_files)
dex_mapping_df.reset_index().set_index(["filename", "hash_key"])

dex_file_playstore  \
filename                                           hash_key                                                                
signal-android_v7.30.2.tar.gz                      f32782128a6145b054f1a5c65cb05902ed002e71033ce1f...       classes2.dex   
                                                   a4decdb3a060d81ecc3f59f8ee02e625629ed2b17546d96...       classes3.dex   
                                                   c061b0748ea8654fa3a585f4e11557e35889517eecb732f...       classes4.dex   
                                                   4d8d31008edf2ebb087e457c115b6f76878b6af3ab0aa8d...       classes6.dex   
dfstest-signal-android-ctime-reversed_v7.37.2_0... c277d27d1d609126923d66ed711ef6b58d9b7f27edf1bac...       classes2.dex   
...                                                                                                                  ...   
signal-android-ctime-sort_v7.37.2.tar.gz           7ec6b5a5f1a5fe7cf4ab84c422245dacb8c989ded81cc6b...       classes6.dex   
signal-android-ctime-reversed_v7.28.4_03.tar.gz    4ab8b39fa16eacf5ede73ac5e4ee21b6a76787b7dc453cf...       classes2.dex   
                                                   0f70ed35963ff35aa0a6c860aa89d5f6c9bef5f9b1b3d1b...       classes3.dex   
                                                   9b27d12a5cef9990c2d0b674489c476ff490749dd109d7c...       classes4.dex   
                                                   1ea7370e3a24cfe9e9c2a84853546a54d6faa0a95c0f7c3...       classes6.dex   

                                                                                                      dex_file_local  
filename                                           hash_key                                                           
signal-android_v7.30.2.tar.gz                      f32782128a6145b054f1a5c65cb05902ed002e71033ce1f...   classes3.dex  
                                                   a4decdb3a060d81ecc3f59f8ee02e625629ed2b17546d96...   classes4.dex  
                                                   c061b0748ea8654fa3a585f4e11557e35889517eecb732f...   classes6.dex  
                                                   4d8d31008edf2ebb087e457c115b6f76878b6af3ab0aa8d...   classes2.dex  
dfstest-signal-android-ctime-reversed_v7.37.2_0... c277d27d1d609126923d66ed711ef6b58d9b7f27edf1bac...   classes3.dex  
...                                                                                                              ...  
signal-android-ctime-sort_v7.37.2.tar.gz           7ec6b5a5f1a5fe7cf4ab84c422245dacb8c989ded81cc6b...   classes2.dex  
signal-android-ctime-reversed_v7.28.4_03.tar.gz    4ab8b39fa16eacf5ede73ac5e4ee21b6a76787b7dc453cf...   classes3.dex  
                                                   0f70ed35963ff35aa0a6c860aa89d5f6c9bef5f9b1b3d1b...   classes4.dex  
                                                   9b27d12a5cef9990c2d0b674489c476ff490749dd109d7c...   classes6.dex  
                                                   1ea7370e3a24cfe9e9c2a84853546a54d6faa0a95c0f7c3...   classes2.dex  

[84 rows x 2 columns]

In [10]:
# Step 1: Check if the mapping from dex_file_playstore -> dex_file_local is consistent
playstore_to_local_mapping = (
    dex_mapping_df[["dex_file_playstore", "dex_file_local"]]
    .groupby("dex_file_playstore")["dex_file_local"]
    .nunique()
)

# Find any dex_file_playstore that maps to more than one dex_file_local
inconsistent_playstore_mapping = playstore_to_local_mapping[
    playstore_to_local_mapping > 1
]

if inconsistent_playstore_mapping.empty:
    print(
        "The mapping between dex_file_playstore and dex_file_local is consistent for all playstore files!"
    )
else:
    print("Inconsistent mappings for the following playstore files:")
    print(inconsistent_playstore_mapping)

# Step 2: Check if the mapping from dex_file_local -> dex_file_playstore is consistent
local_to_playstore_mapping = (
    dex_mapping_df[["dex_file_local", "dex_file_playstore"]]
    .groupby("dex_file_local")["dex_file_playstore"]
    .nunique()
)

# Find any dex_file_local that maps to more than one dex_file_playstore
inconsistent_local_mapping = local_to_playstore_mapping[local_to_playstore_mapping > 1]

if inconsistent_local_mapping.empty:
    print(
        "The mapping between dex_file_local and dex_file_playstore is consistent for all local files!"
    )
else:
    print("Inconsistent mappings for the following local files:")
    print(inconsistent_local_mapping)

The mapping between dex_file_playstore and dex_file_local is consistent for all playstore files!
The mapping between dex_file_local and dex_file_playstore is consistent for all local files!


In [11]:
# Filter the final_df to get only the 'differing_dexes' category
differing_dexes_df = df[df.index.get_level_values("category") == "differing_dexes"]

# Group by 'filename' and 'dex_file', then count occurrences
dex_file_counts = (
    differing_dexes_df.groupby(["filename", "dex_file"])
    .size()
    .reset_index(name="count")
)

# Sort by the 'count' column for better visibility
dex_file_counts_sorted = dex_file_counts.sort_values(by="count", ascending=False)

dex_file_counts_sorted["parsed"] = dex_file_counts_sorted["filename"].apply(
    parameters_from_tar_filename
)
# Split the 'parsed' column into separate columns
dex_file_counts_sorted[["version", "run", "dfstest", "dfs", "ctime", "reverse"]] = (
    pd.DataFrame(
        dex_file_counts_sorted["parsed"].tolist(), index=dex_file_counts_sorted.index
    )
)

# Now set the hierarchical index using 'dex_file' and the split columns from 'parsed'
dex_file_counts_sorted.set_index(
    ["version", "dfstest", "dfs", "ctime", "reverse", "run"], inplace=True
)
# todo: do we want dex_file to be the first dimension?
dex_file_counts_sorted.drop(columns=["filename", "parsed", "count"], inplace=True)

In [12]:
show_full_df(dex_file_counts_sorted)

dex_file
version dfstest dfs   ctime reverse run                         
7.28.4  True    True  True  True    1         local->classes.dex
                                    1        local->classes2.dex
                                    1        local->classes3.dex
                                    1        local->classes4.dex
                                    1        local->classes5.dex
                                    1        local->classes6.dex
                                    1     playstore->classes.dex
                                    1    playstore->classes2.dex
                                    1    playstore->classes3.dex
                                    1    playstore->classes4.dex
                                    1    playstore->classes5.dex
                                    1    playstore->classes6.dex
                                    1    playstore->classes7.dex
                                    2         local->classes.dex
                                    2        local->classes2.dex
                                    2        local->classes3.dex
                                    2        local->classes4.dex
                                    2        local->classes5.dex
                                    2        local->classes6.dex
                                    2     playstore->classes.dex
                                    2    playstore->classes2.dex
                                    2    playstore->classes3.dex
                                    2    playstore->classes4.dex
                                    2    playstore->classes5.dex
                                    2    playstore->classes6.dex
                                    2    playstore->classes7.dex
7.37.2  True    True  True  True    1    playstore->classes7.dex
                                    2    playstore->classes7.dex
7.28.4  True    True  True  False   1    playstore->classes7.dex
                                    2    playstore->classes7.dex
7.37.2  True    True  True  False   1    playstore->classes7.dex
                                    2    playstore->classes7.dex
7.30.2  False   True  False True    1    playstore->classes7.dex
7.37.2  False   True  False True    1    playstore->classes7.dex
7.30.2  False   True  False False   1    playstore->classes7.dex
7.37.2  False   True  False False   1    playstore->classes7.dex
7.28.4  False   True  True  True    1         local->classes.dex
                                    1        local->classes2.dex
                                    1        local->classes3.dex
                                    1        local->classes4.dex
                                    1        local->classes5.dex
                                    1        local->classes6.dex
                                    1     playstore->classes.dex
                                    1    playstore->classes2.dex
                                    1    playstore->classes3.dex
                                    1    playstore->classes4.dex
                                    1    playstore->classes5.dex
                                    1    playstore->classes6.dex
                                    1    playstore->classes7.dex
                                    2    playstore->classes7.dex
                                    3    playstore->classes7.dex
7.30.2  False   True  True  True    1    playstore->classes7.dex
7.36.2  False   True  True  True    1    playstore->classes7.dex
7.37.2  False   True  True  True    1    playstore->classes7.dex
7.28.4  False   True  True  False   1         local->classes.dex
                                    1        local->classes2.dex
                                    1        local->classes3.dex
                                    1        local->classes4.dex
                                    1        local->classes5.dex
                                    1        local->classes6.dex
                          

## Check 2: remaining files

The following files have different sets of hashes for the `play` and `local` build.

In [13]:
all_filenames = set(df.index.get_level_values("filename").tolist())

In [14]:
same_hashes_df = dex_mapping_df.reset_index().set_index(["filename", "hash_key"])
filenames_same_hashes = set(same_hashes_df.index.get_level_values("filename").tolist())

In [15]:
all_filenames - filenames_same_hashes

{'dfstest-signal-android-ctime-reversed_v7.28.4_01.tar.gz',
 'dfstest-signal-android-ctime-reversed_v7.28.4_02.tar.gz',
 'signal-android-ctime-reversed_v7.28.4_01.tar.gz',
 'signal-android-ctime-sort_v7.28.4.tar.gz'}